# LLM Prompt Speed Comparison

This notebook compares the speed of running BAGEL prompts across different models:
- **Local Ollama models** (gpt-oss and others)
- **Remote OpenAI API models**

Run the cells you need based on your environment.

## Setup and Imports

In [2]:
import json
import time
import re
from typing import List, Tuple, Dict, Any
from ollama import Client
from openai import OpenAI
from scripts.response_schema import Response
import pandas as pd
import numpy as np

## Load Sample Prompts

Load prompts from the OpenAI batch files or parsed inputs directory.

In [3]:
def load_prompts_from_jsonl(file_path: str, max_prompts: int = 5) -> List[Dict]:
    """
    Load prompts from a JSONL file.
    For OpenAI batch files, extract the prompt from body.messages[0].content
    For bodies files, use the 'prompt' field directly
    """
    prompts = []
    with open(file_path) as f:
        for i, line in enumerate(f):
            if i >= max_prompts:
                break
            if line.strip():
                data = json.loads(line)
                # Check if it's an OpenAI batch format
                if 'body' in data and 'messages' in data['body']:
                    prompt = data['body']['messages'][0]['content']
                    idx = data.get('custom_id', f'{i}')
                # Check if it's a bodies format
                elif 'prompt' in data:
                    prompt = data['prompt']
                    idx = data.get('index', i)
                else:
                    continue
                prompts.append({'index': idx, 'prompt': prompt})
    return prompts

# Load sample prompts - adjust path as needed
PROMPT_FILE = 'data/run_5/open_ai_batches_10/openai_batch_o3-mini_bodies_10.jsonl'
# Alternative: use parsed inputs
# PROMPT_FILE = 'data/run_5/parsed_inputs/bodies_10.jsonl'

sample_prompts = load_prompts_from_jsonl(PROMPT_FILE, max_prompts=5)
print(f"Loaded {len(sample_prompts)} sample prompts")
print(f"First prompt preview (first 200 chars): {sample_prompts[0]['prompt'][:200]}...")

Loaded 5 sample prompts
First prompt preview (first 200 chars): You are an expert engine for analyzing biomedical vocabularies and ontologies. Your task is to determine the semantic relationship between a query term, used within the context of a scientific abstrac...


## Helper Functions for Timing

In [4]:
def time_prompts(prompts: List[Dict], run_function, model_name: str) -> pd.DataFrame:
    """
    Time a list of prompts and return results as a DataFrame.
    
    Args:
        prompts: List of prompt dicts with 'index' and 'prompt' keys
        run_function: Function that takes (idx, prompt) and returns (message_content, response)
        model_name: Name of the model being tested
    
    Returns:
        DataFrame with timing results
    """
    results = []
    
    for prompt_data in prompts:
        idx = prompt_data['index']
        prompt = prompt_data['prompt']
        
        print(f"Running prompt {idx}...", end=' ')
        start_time = time.time()
        
        try:
            message_content, response = run_function(idx, prompt)
            elapsed = time.time() - start_time
            
            response_length = len(message_content) if message_content else 0
            success = True
            error_msg = None
            
            print(f"✓ {elapsed:.2f}s ({response_length} chars)")
            
        except Exception as e:
            elapsed = time.time() - start_time
            response_length = 0
            success = False
            error_msg = str(e)
            print(f"✗ {elapsed:.2f}s (ERROR: {error_msg})")
        
        results.append({
            'model': model_name,
            'index': idx,
            'time_seconds': elapsed,
            'response_length': response_length,
            'success': success,
            'error': error_msg
        })
    
    return pd.DataFrame(results)

def summarize_results(df: pd.DataFrame) -> None:
    """
    Print summary statistics for timing results.
    """
    print("\n" + "="*60)
    print(f"SUMMARY: {df['model'].iloc[0]}")
    print("="*60)
    
    successful = df[df['success'] == True]
    
    print(f"Total prompts: {len(df)}")
    print(f"Successful: {len(successful)} ({len(successful)/len(df)*100:.1f}%)")
    print(f"Failed: {len(df) - len(successful)}")
    
    if len(successful) > 0:
        print(f"\nTiming Statistics (successful prompts):")
        print(f"  Mean: {successful['time_seconds'].mean():.2f}s")
        print(f"  Median: {successful['time_seconds'].median():.2f}s")
        print(f"  Min: {successful['time_seconds'].min():.2f}s")
        print(f"  Max: {successful['time_seconds'].max():.2f}s")
        print(f"  Std Dev: {successful['time_seconds'].std():.2f}s")
        
        print(f"\nResponse Length Statistics:")
        print(f"  Mean: {successful['response_length'].mean():.0f} chars")
        print(f"  Median: {successful['response_length'].median():.0f} chars")
    
    print("="*60)

---
## Local Ollama Testing

**Run this cell to test local ollama models (e.g., gpt-oss)**

In [5]:
def run_ollama_prompt(idx: int, prompt: str, model: str = "gpt-oss", 
                      format: bool = False, url: str = "local") -> Tuple[str, dict]:
    """
    Run a prompt using the Ollama API.
    Adapted from scripts/run_ollama.py
    
    Args:
        idx: Index of the prompt
        prompt: The prompt text
        model: Ollama model name
        format: Whether to use structured JSON output (requires Response schema)
        url: "local" for localhost:11434 or custom URL
    
    Returns:
        Tuple of (message_content, full_response)
    """
    host = "http://localhost:11434" if url == "local" else url
    client = Client(host=host)
    
    chat_kwargs: Dict[str, Any] = {
        'model': model,
        'messages': [{'role': 'user', 'content': prompt}]
    }
    
    if format:
        chat_kwargs['format'] = Response.model_json_schema()
        response = client.chat(**chat_kwargs)
        message_content = response['message']['content']
    else:
        response = client.chat(**chat_kwargs)
        message_content = response['message']['content']
        # Try to extract JSON from markdown code blocks
        match = re.search(r'```json\s*(.*?)```', message_content, re.DOTALL)
        if match:
            message_content = match.group(1).strip()
    
    return message_content, response

# Configure ollama model to test
OLLAMA_MODEL = "gpt-oss"  # Change this to test other models
OLLAMA_FORMAT = False  # Set to True if model supports structured output
OLLAMA_URL = "local"  # Change for remote ollama servers

# Create wrapper function for timing
def run_ollama_wrapper(idx, prompt):
    return run_ollama_prompt(idx, prompt, model=OLLAMA_MODEL, 
                            format=OLLAMA_FORMAT, url=OLLAMA_URL)

# Run timing test
print(f"Testing Ollama model: {OLLAMA_MODEL}")
print(f"Running {len(sample_prompts)} prompts...\n")

ollama_results = time_prompts(sample_prompts, run_ollama_wrapper, OLLAMA_MODEL)
summarize_results(ollama_results)

# Display detailed results
ollama_results

Testing Ollama model: gpt-oss
Running 5 prompts...

Running prompt 0_o3-mini... ✓ 82.98s (6344 chars)
Running prompt 1_o3-mini... ✓ 51.78s (4893 chars)
Running prompt 2_o3-mini... ✓ 59.88s (6054 chars)
Running prompt 3_o3-mini... ✓ 59.46s (4682 chars)
Running prompt 4_o3-mini... ✓ 84.89s (7099 chars)

SUMMARY: gpt-oss
Total prompts: 5
Successful: 5 (100.0%)
Failed: 0

Timing Statistics (successful prompts):
  Mean: 67.80s
  Median: 59.88s
  Min: 51.78s
  Max: 84.89s
  Std Dev: 15.10s

Response Length Statistics:
  Mean: 5814 chars
  Median: 6054 chars


,model,index,time_seconds,response_length,success,error
0,gpt-oss,0_o3-mini,82.978260,6344,True,None
1,gpt-oss,1_o3-mini,51.778463,4893,True,None
2,gpt-oss,2_o3-mini,59.879057,6054,True,None
3,gpt-oss,3_o3-mini,59.455656,4682,True,None
4,gpt-oss,4_o3-mini,84.888819,7099,True,None


---
## OpenAI API Testing

**Run this cell to test OpenAI models**

Make sure you have set your `OPENAI_API_KEY` environment variable.

In [ ]:
def run_openai_prompt(idx: int, prompt: str, model: str = "gpt-4o-mini", 
                      response_format: str = None) -> Tuple[str, dict]:
    """
    Run a prompt using the OpenAI API.
    
    Args:
        idx: Index of the prompt
        prompt: The prompt text
        model: OpenAI model name (e.g., 'gpt-4o-mini', 'gpt-4o', 'o1-mini')
        response_format: Optional response format ("json_object" for JSON mode)
    
    Returns:
        Tuple of (message_content, full_response)
    """
    client = OpenAI()  # Uses OPENAI_API_KEY environment variable
    
    messages = [{'role': 'user', 'content': prompt}]
    
    kwargs = {
        'model': model,
        'messages': messages
    }
    
    # Add JSON mode if requested (not supported by all models)
    if response_format == "json_object":
        kwargs['response_format'] = {'type': 'json_object'}
    
    response = client.chat.completions.create(**kwargs)
    
    message_content = response.choices[0].message.content
    
    # Convert response to dict for consistency with ollama interface
    response_dict = {
        'id': response.id,
        'model': response.model,
        'usage': {
            'prompt_tokens': response.usage.prompt_tokens,
            'completion_tokens': response.usage.completion_tokens,
            'total_tokens': response.usage.total_tokens
        },
        'message': {
            'role': response.choices[0].message.role,
            'content': message_content
        }
    }
    
    return message_content, response_dict

# Configure OpenAI model to test
OPENAI_MODEL = "gpt-4o-mini"  # Options: gpt-4o-mini, gpt-4o, o1-mini, o3-mini, etc.
OPENAI_JSON_MODE = None  # Set to "json_object" for JSON mode

# Create wrapper function for timing
def run_openai_wrapper(idx, prompt):
    return run_openai_prompt(idx, prompt, model=OPENAI_MODEL, 
                            response_format=OPENAI_JSON_MODE)

# Run timing test
print(f"Testing OpenAI model: {OPENAI_MODEL}")
print(f"Running {len(sample_prompts)} prompts...\n")

openai_results = time_prompts(sample_prompts, run_openai_wrapper, OPENAI_MODEL)
summarize_results(openai_results)

# Display detailed results including token usage
openai_results

---
## Alternative OpenAI Model Testing

**Run this cell to test a different OpenAI model (e.g., GPT-4o or o3-mini)**

In [ ]:
# Configure alternative OpenAI model
ALT_OPENAI_MODEL = "gpt-4o"  # or "o1-mini", "o3-mini", etc.
ALT_OPENAI_JSON_MODE = None

# Create wrapper function
def run_alt_openai_wrapper(idx, prompt):
    return run_openai_prompt(idx, prompt, model=ALT_OPENAI_MODEL, 
                            response_format=ALT_OPENAI_JSON_MODE)

# Run timing test
print(f"Testing OpenAI model: {ALT_OPENAI_MODEL}")
print(f"Running {len(sample_prompts)} prompts...\n")

alt_openai_results = time_prompts(sample_prompts, run_alt_openai_wrapper, ALT_OPENAI_MODEL)
summarize_results(alt_openai_results)

alt_openai_results

---
## Compare All Results

**Run this cell after running tests to compare models**

In [ ]:
# Combine all results that have been run
all_results = []

if 'ollama_results' in locals():
    all_results.append(ollama_results)
if 'openai_results' in locals():
    all_results.append(openai_results)
if 'alt_openai_results' in locals():
    all_results.append(alt_openai_results)

if len(all_results) > 1:
    combined_df = pd.concat(all_results, ignore_index=True)
    
    # Compare successful runs only
    successful = combined_df[combined_df['success'] == True]
    
    print("\n" + "="*60)
    print("MODEL COMPARISON (Successful Runs Only)")
    print("="*60)
    
    comparison = successful.groupby('model').agg({
        'time_seconds': ['count', 'mean', 'median', 'min', 'max', 'std'],
        'response_length': ['mean', 'median']
    }).round(2)
    
    print(comparison)
    
    # Bar chart comparison
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Mean time comparison
    mean_times = successful.groupby('model')['time_seconds'].mean().sort_values()
    mean_times.plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_xlabel('Mean Time (seconds)')
    axes[0].set_title('Average Response Time by Model')
    axes[0].grid(axis='x', alpha=0.3)
    
    # Response length comparison
    mean_lengths = successful.groupby('model')['response_length'].mean().sort_values()
    mean_lengths.plot(kind='barh', ax=axes[1], color='coral')
    axes[1].set_xlabel('Mean Response Length (characters)')
    axes[1].set_title('Average Response Length by Model')
    axes[1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
else:
    print("Run at least 2 model tests to see comparison.")

---
## Export Results

**Optional: Save results to CSV for further analysis**

In [ ]:
# Export all results to CSV
if len(all_results) > 0:
    output_file = 'prompt_speed_comparison_results.csv'
    combined_df.to_csv(output_file, index=False)
    print(f"Results saved to {output_file}")
else:
    print("No results to export. Run some model tests first.")